# Trace Analysis

In [1]:
import json
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from trace_viewer import show_trace

TRACES_DIR = Path("../traces")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 50)

## Dashboard

In [2]:
results = pd.read_csv(TRACES_DIR / "results.csv")
results["passed"] = results["passed"].astype(bool)
for col in ["model_time_s", "tool_time_s"]:
    if col not in results.columns:
        results[col] = float("nan")
print(f"{len(results)} total results across {results['timestamp'].nunique()} runs")

132 total results across 9 runs


In [3]:
# Run-over-run summary (last 10 runs)
runs = results.groupby("timestamp").agg(
    passed=("passed", "sum"),
    questions=("passed", "count"),
    pass_rate=("passed", "mean"),
    avg_tool_calls=("tool_calls", "mean"),
    avg_model_time=("model_time_s", "mean"),
    avg_tool_time=("tool_time_s", "mean"),
    avg_wall_time=("wall_time_s", "mean"),
    avg_tokens=("total_tokens", "mean"),
    avg_cost=("cost_usd", "mean"),
    total_cost=("cost_usd", "sum"),
).tail(10)
runs["pass_rate"] = runs["pass_rate"].map("{:.0%}".format)
runs["avg_tool_calls"] = runs["avg_tool_calls"].round(1)
runs["avg_model_time"] = runs["avg_model_time"].round(1)
runs["avg_tool_time"] = runs["avg_tool_time"].round(1)
runs["avg_wall_time"] = runs["avg_wall_time"].round(1)
runs["avg_tokens"] = runs["avg_tokens"].astype(int)
runs["avg_cost"] = runs["avg_cost"].map("${:.4f}".format)
runs["total_cost"] = runs["total_cost"].map("${:.4f}".format)
runs

,passed,questions,pass_rate,avg_tool_calls,avg_model_time,avg_tool_time,avg_wall_time,avg_tokens,avg_cost,total_cost
timestamp,,,,,,,,,,
2026-03-13 22:40,12,14,86%,5.9,NaN,NaN,24.1,44933,$0.0099,$0.1390
2026-03-14 07:42,12,14,86%,6.9,NaN,NaN,20.1,56346,$0.0170,$0.2386
2026-03-15 08:13,12,14,86%,13.6,NaN,NaN,91.7,80279,$0.0317,$0.4441
2026-03-15 15:05,13,14,93%,8.1,NaN,NaN,34.6,49770,$0.0151,$0.2119
2026-03-15 17:13,14,14,100%,6.0,NaN,NaN,27.6,23786,$0.0089,$0.1252
2026-03-15 17:38,13,14,93%,7.0,NaN,NaN,40.4,36117,$0.0135,$0.1888
2026-03-15 20:12,14,14,100%,5.5,NaN,NaN,29.9,28908,$0.0113,$0.1584
2026-03-16 09:41,14,15,93%,4.9,NaN,NaN,38.0,25900,$0.0104,$0.1564
2026-03-16 15:49,16,19,84%,5.6,35.4,6.4,41.8,34901,$0.0138,$0.2622


In [4]:
# Select a run to inspect (default: latest)
RUN = results["timestamp"].max()

run = results[results["timestamp"] == RUN]
passed = run["passed"].sum()
total = len(run)
print(f"Selected: {RUN} | {passed}/{total} passed")

display_cols = ["workspace", "question", "passed", "tool_calls", "model_time_s", "tool_time_s", "wall_time_s", "failed_assertions"]
run[display_cols].sort_values("passed")

Selected: 2026-03-16 15:49 | 16/19 passed


,workspace,question,passed,tool_calls,model_time_s,tool_time_s,wall_time_s,failed_assertions
127,sec-10k,Which company in the dataset had the highest net income?,False,1,2.60,2.32,4.92,contains 'alphabet'; min_tool_calls >= 3; has_answer
124,sec-10k,What was Amazon's net income in fiscal year 2025?,False,10,75.62,5.41,81.03,"contains_any ['77,670,000,000', '77.7 billion', '77.7B',..."
120,origin-of-species,What examples of variation under domestication does Darw...,False,10,16.98,6.81,23.79,"contains_any ['pigeon', 'dog', 'cattle', 'horse', 'sheep..."
113,federalist-papers,What does Federalist No. 10 argue about factions?,True,4,20.30,9.16,29.46,NaN
129,world-factbook,What is the population of Japan?,True,7,23.22,4.39,27.61,NaN
128,sec-10k,Which company in the dataset had the highest total assets?,True,6,36.19,35.73,71.92,NaN
126,sec-10k,Which company in the dataset had the highest total revenue?,True,9,88.92,7.02,95.94,NaN
125,sec-10k,Which company had higher total revenue in fiscal year 20...,True,11,142.90,6.05,148.95,NaN
123,sherlock-holmes,What methods does Holmes use across the stories?,True,5,33.94,9.06,43.00,NaN
122,sherlock-holmes,How does Holmes solve the case in The Speckled Band?,True,2,20.70,1.28,21.98,NaN


In [5]:
# Failures for the selected run
failures = run[~run["passed"]]
if len(failures):
    print(f"{len(failures)} failures:")
    display(failures[["workspace", "question", "tool_calls", "failed_assertions", "error"]])
else:
    print("No failures!")

3 failures:


,workspace,question,tool_calls,failed_assertions,error
120,origin-of-species,What examples of variation under domestication does Darw...,10,"contains_any ['pigeon', 'dog', 'cattle', 'horse', 'sheep...","litellm.APIError: APIError: OpenrouterException - {""erro..."
124,sec-10k,What was Amazon's net income in fiscal year 2025?,10,"contains_any ['77,670,000,000', '77.7 billion', '77.7B',...",NaN
127,sec-10k,Which company in the dataset had the highest net income?,1,contains 'alphabet'; min_tool_calls >= 3; has_answer,"litellm.APIError: APIError: OpenrouterException - {""erro..."


## Trace Viewer

In [6]:
# Build trace index: model → [(label, path), ...]
trace_index: dict[str, list[tuple[str, Path]]] = {}
for model_dir in sorted(TRACES_DIR.iterdir()):
    if not model_dir.is_dir() or model_dir.name.endswith(".csv"):
        continue
    entries = []
    for ws_dir in sorted(model_dir.iterdir()):
        if not ws_dir.is_dir():
            continue
        for f in sorted(ws_dir.glob("*.json")):
            data = json.loads(f.read_text())
            status = "PASS" if data["passed"] else "FAIL"
            entries.append((f"[{status}] {ws_dir.name} / {f.stem}", f))
    if entries:
        trace_index[model_dir.name] = entries

model_dropdown = widgets.Dropdown(
    options=list(trace_index.keys()),
    value=list(trace_index.keys())[-1],
    description="Model:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="60%"),
)
question_dropdown = widgets.Dropdown(
    description="Question:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="90%"),
)
output = widgets.Output()


def _update_questions(_change=None):
    entries = trace_index[model_dropdown.value]
    question_dropdown.options = [label for label, _ in entries]
    question_dropdown.value = question_dropdown.options[0]


def _show_trace(_change=None):
    entries = trace_index[model_dropdown.value]
    label_to_path = dict(entries)
    path = label_to_path.get(question_dropdown.value)
    if not path:
        return
    data = json.loads(path.read_text())
    with output:
        output.clear_output(wait=True)
        show_trace(data, max_chars=2000)


model_dropdown.observe(_update_questions, names="value")
question_dropdown.observe(_show_trace, names="value")

_update_questions()
display(model_dropdown, question_dropdown, output)
_show_trace()

Dropdown(description='Model:', index=1, layout=Layout(width='60%'), options=('openrouter-deepseek-deepseek-v3-…

Dropdown(description='Question:', layout=Layout(width='90%'), options=('[PASS] federalist-papers / how-do-hami…

Output()